# Resolution Sensitivity + Model Misspecification Study

## Design
**2 (true_smooth) × 4 (resolution) × 2 (fit_smooth) = 16 fits**

| true_smooth | fit_smooth | status |
|---|---|---|
| 0.5 | 0.5 | correctly specified |
| 0.5 | 1.5 | **misspecified** |
| 1.5 | 0.5 | **misspecified** |
| 1.5 | 1.5 | correctly specified |

Resolutions: **x8** (coarsest) → **x4** → **x2** → **x1** (original GEMS grid spacing)

## Kernel formulas (d = phi2 × anisotropic distance)
- **smooth=0.5** (Matérn 1/2): `C = σ² exp(−d)`
- **smooth=1.5** (Matérn 3/2): `C = σ² (1+d) exp(−d)`

## Hybrid Vecchia conditioning sets
| Set | Time | Content |
|---|---|---|
| A | t | spatial NN × 20 |
| B | t−1 | same-loc (1) + local NN (16) + shifted upstream NN (2) |
| C | t−2 | same-loc (1) + local NN (12) + shifted 2× upstream NN (2) |

In [ ]:
import gc, os, sys, time, random, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from sklearn.neighbors import BallTree

LOCAL_SRC  = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import configuration as config
from GEMS_TCO import orderings as _orderings
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit
from GEMS_TCO.data_loader import load_data_dynamic_processed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float64
print(f"Device: {DEVICE}")

In [ ]:
# ---- base grid spacings (x1 = original GEMS resolution) ----
DELTA_LAT_BASE = 0.044   # degrees lat per cell
DELTA_LON_BASE = 0.063   # degrees lon per cell
T_STEPS        = 8

# ---- region ----
LAT_RANGE = [-3.0,  2.0]
LON_RANGE = [121.0, 131.0]

# ---- GEMS obs pattern ----
YEAR  = "2023"
MONTH = 7

# ---- high-res simulation grid (lat×100, lon×10 finer than base) ----
HR_LAT_FACTOR = 100
HR_LON_FACTOR = 10

# ---- resolution multipliers (R): spacing = R × DELTA_BASE ----
# x8 = coarsest (fewest obs); x1 = original (most obs)
RESOLUTION_MULTS = [8, 4, 2, 1]

# ---- hybrid Vecchia spec ----
# Set A: 20 spatial NN at t
# Set B: 1 same-loc + 16 local NN + 2 fresh shifted NN  at t-1
# Set C: 1 same-loc + 12 local NN + 2 fresh shifted NN  at t-2
MM_COND_NUMBER  = 100
NHEADS          = 0
DAILY_STRIDE    = 2
LIMIT_A         = 20
LIMIT_B_LOCAL   = 16
LIMIT_C_LOCAL   = 12
LAG1_FRESH      = 2
LAG2_FRESH      = 2
LAG1_LON_OFFSET = 0.063   # predicted upstream shift (degrees east)

# ---- optimizer ----
LBFGS_STEPS = 6
LBFGS_EVAL  = 20
LBFGS_HIST  = 10
LBFGS_LR    = 1.0

# ---- true parameters (identical for both smoothness scenarios) ----
TRUE_DICT = {
    "sigmasq":    10.0,
    "range_lat":   0.5,
    "range_lon":   0.6,
    "range_time":  2.5,
    "advec_lat":   0.08,
    "advec_lon":  -0.16,
    "nugget":      1.2,
}
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time",
            "advec_lat", "advec_lon", "nugget"]
P_DISP   = [r"$\sigma^2$", r"$r_{lat}$", r"$r_{lon}$", r"$r_{time}$",
            r"$\alpha_{lat}$", r"$\alpha_{lon}$", "nugget"]

TRUE_SMOOTHS = [0.5, 1.5]
FIT_SMOOTHS  = [0.5, 1.5]

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Constants ready.", TRUE_DICT)

## Helper Functions

In [ ]:
# ---- parameter reparametrization ----

def true_to_log_params(d):
    """Physical → log-reparametrized (phi1..phi4, advec, log_nugget)."""
    phi2 = 1.0 / d["range_lon"]
    phi1 = d["sigmasq"] * phi2
    phi3 = (d["range_lon"] / d["range_lat"]) ** 2
    phi4 = (d["range_lon"] / d["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d["advec_lat"], d["advec_lon"], np.log(d["nugget"])]


def backmap_params(out_params):
    """Log-reparametrized → physical."""
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq":    np.exp(p[0]) / phi2,
        "range_lat":  rlon / phi3 ** 0.5,
        "range_lon":  rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat":  p[4],
        "advec_lon":  p[5],
        "nugget":     np.exp(p[6]),
    }


# ---- smooth-aware covariance for circulant embedding ----

def get_covariance_on_grid(lx, ly, lt, params, smooth=0.5):
    """
    Stationary covariance at spatial/temporal lag (lx, ly, lt).
    Used for circulant embedding field simulation.

    params = [log_phi1, log_phi2, log_phi3, log_phi4, advec_lat, advec_lon, log_nugget]
    d = sqrt(phi3*(lx - advec_lat*lt)^2 + (ly - advec_lon*lt)^2 + phi4*lt^2)
    smooth=0.5: C = (phi1/phi2) * exp(-phi2*d)
    smooth=1.5: C = (phi1/phi2) * (1 + phi2*d) * exp(-phi2*d)
    """
    params   = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat    = lx - params[4] * lt
    u_lon    = ly - params[5] * lt
    d        = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    scaled_d = d * phi2
    if smooth == 0.5:
        return (phi1 / phi2) * torch.exp(-scaled_d)
    else:  # 1.5
        return (phi1 / phi2) * (1.0 + scaled_d) * torch.exp(-scaled_d)


def generate_field_values(lats_hr, lons_hr, t_steps, params, dlat, dlon, smooth=0.5):
    """Simulate a GP field via circulant (spectral) embedding."""
    cpu = torch.device("cpu")
    f32 = torch.float32
    nx, ny, nt = len(lats_hr), len(lons_hr), t_steps
    px, py, pt = 2 * nx, 2 * ny, 2 * nt
    lx = torch.arange(px, device=cpu, dtype=f32) * dlat
    lx[px // 2:] -= px * dlat
    ly = torch.arange(py, device=cpu, dtype=f32) * dlon
    ly[py // 2:] -= py * dlon
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    params_cpu   = params.cpu().float()
    lx_g, ly_g, lt_g = torch.meshgrid(lx, ly, lt, indexing="ij")
    cov  = get_covariance_on_grid(lx_g, ly_g, lt_g, params_cpu, smooth=smooth)
    spec = torch.fft.fftn(cov)
    spec.real = torch.clamp(spec.real, min=0)   # enforce non-negative spectrum
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(spec.real) * noise).real[:nx, :ny, :nt]
    return field.to(dtype=DTYPE, device=DEVICE)


# ---- observation-to-grid mapping ----

def apply_step3_1to1(src_np_valid, grid_coords_np, grid_tree, dlat_half, dlon_half):
    """
    1-to-1 assignment: each grid cell gets at most one observation (the nearest).
    Tolerance: ±dlat_half in lat, ±dlon_half in lon.
    Returns assignment[cell_j] = obs_i that wins, or -1 if no obs within tolerance.
    """
    n_grid = len(grid_coords_np)
    if len(src_np_valid) == 0:
        return np.full(n_grid, -1, dtype=np.int64)
    dist_to_cell, cell_for_obs = grid_tree.query(np.radians(src_np_valid), k=1)
    dist_to_cell = dist_to_cell.flatten()
    cell_for_obs = cell_for_obs.flatten()
    assignment   = np.full(n_grid, -1, dtype=np.int64)
    best_dist    = np.full(n_grid, np.inf)
    for obs_i, (cell_j, dist) in enumerate(zip(cell_for_obs, dist_to_cell)):
        if dist < best_dist[cell_j]:
            assignment[cell_j] = obs_i
            best_dist[cell_j]  = dist
    # remove matches that fall outside the cell tolerance
    filled = assignment >= 0
    if filled.any():
        win_obs  = assignment[filled]
        lat_diff = np.abs(src_np_valid[win_obs, 0] - grid_coords_np[filled, 0])
        lon_diff = np.abs(src_np_valid[win_obs, 1] - grid_coords_np[filled, 1])
        too_far  = (lat_diff > dlat_half) | (lon_diff > dlon_half)
        assignment[np.where(filled)[0][too_far]] = -1
    return assignment


def precompute_mapping_indices(ref_day_map, lats_hr, lons_hr,
                               grid_coords, sorted_keys, dlat_half, dlon_half):
    """
    For each time step:
      step3_assignment: grid cell → winning obs index (or -1)
      hr_idx:           obs → nearest high-res grid cell (for reading the simulated field)
      src_locs:         actual (lat, lon) of each obs
    """
    hr_lat_g, hr_lon_g = torch.meshgrid(lats_hr, lons_hr, indexing="ij")
    hr_coords_np  = torch.stack([hr_lat_g.flatten(), hr_lon_g.flatten()], dim=1).cpu().numpy()
    hr_tree       = BallTree(np.radians(hr_coords_np), metric="haversine")
    grid_coords_np = grid_coords.cpu().numpy()
    n_grid         = len(grid_coords_np)
    grid_tree      = BallTree(np.radians(grid_coords_np), metric="haversine")

    s3_list, hr_list, src_list = [], [], []

    for key in sorted_keys:
        df = ref_day_map.get(key)
        if df is None or len(df) == 0:
            s3_list.append(np.full(n_grid, -1, dtype=np.int64))
            hr_list.append(torch.zeros(0, dtype=torch.long, device=DEVICE))
            src_list.append(torch.zeros((0, 2), dtype=DTYPE, device=DEVICE))
            continue
        src_np       = df[["Source_Latitude", "Source_Longitude"]].values
        valid_mask   = ~np.isnan(src_np).any(axis=1)
        src_np_valid = src_np[valid_mask]
        assignment   = apply_step3_1to1(src_np_valid, grid_coords_np, grid_tree,
                                        dlat_half, dlon_half)
        s3_list.append(assignment)
        if len(src_np_valid) > 0:
            _, hi = hr_tree.query(np.radians(src_np_valid), k=1)
            hr_list.append(torch.tensor(hi.flatten(), device=DEVICE))
        else:
            hr_list.append(torch.zeros(0, dtype=torch.long, device=DEVICE))
        src_list.append(torch.tensor(src_np_valid, device=DEVICE, dtype=DTYPE))

    return s3_list, hr_list, src_list


def assemble_irregular_map(field, s3_list, hr_list, src_list,
                           sorted_keys, grid_coords, true_params, t_offset=21.0):
    """
    Build the irregular map from a simulated field:
      - Value at each matched grid cell = field[hr_idx] + N(0, nugget)
      - NaN where no observation is assigned
    NOTE: nugget_std = sqrt(exp(true_params[6])) = sqrt(nugget variance)
    """
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid     = grid_coords.shape[0]
    field_flat = field.reshape(-1, T_STEPS)   # (n_hr_cells, T)
    irr_map    = {}
    for t_idx, key in enumerate(sorted_keys):
        t_val    = float(t_offset + t_idx)
        assign   = s3_list[t_idx]
        hr_idx   = hr_list[t_idx]
        src_locs = src_list[t_idx]
        # dummy indicators (7 dummies for T_STEPS=8 minus intercept)
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.full((n_grid, 11), float("nan"), device=DEVICE, dtype=DTYPE)
        rows[:, 3]  = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        if hr_idx.shape[0] > 0:
            gp_vals  = field_flat[hr_idx, t_idx]
            sim_vals = gp_vals + torch.randn(hr_idx.shape[0], device=DEVICE, dtype=DTYPE) * nugget_std
            assign_t = torch.tensor(assign, device=DEVICE, dtype=torch.long)
            filled   = assign_t >= 0
            win_obs  = assign_t[filled]
            rows[filled, 0] = src_locs[win_obs, 0]   # lat
            rows[filled, 1] = src_locs[win_obs, 1]   # lon
            rows[filled, 2] = sim_vals[win_obs]       # value
        irr_map[key] = rows.detach()
    return irr_map


print("Helper functions defined.")

## Load Real GEMS Observation Patterns

In [ ]:
is_amarel   = os.path.exists(config.amarel_data_load_path)
data_path   = config.amarel_data_load_path if is_amarel else config.mac_data_load_path
data_loader = load_data_dynamic_processed(data_path)

# Load one year/month of GEMS obs patterns
df_map, _, _, _ = data_loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)
print(f"Loaded {len(df_map)} time slots for {YEAR}-{MONTH:02d}")

# TCO grid has Source_Latitude/Longitude (actual instrument footprint positions)
yr2      = YEAR[2:]
tco_path = Path(data_path) / f"pickle_{YEAR}" / f"tco_grid_{yr2}_{MONTH:02d}.pkl"
if tco_path.exists():
    with open(tco_path, "rb") as f:
        tco_map = pickle.load(f)
    print(f"Loaded tco_grid: {len(tco_map)} slots")
else:
    tco_map = {}
    print(f"[WARN] tco_grid not found: {tco_path}")

# Pick one 8-step day pattern (first available)
all_sorted = sorted(df_map.keys())
day_keys   = all_sorted[:T_STEPS]
ref_day    = {k: tco_map.get(k.split("_", 2)[-1], pd.DataFrame()) for k in day_keys}
print(f"Day pattern: {day_keys[0]} … {day_keys[-1]}")
for k in day_keys[:2]:
    df_k = ref_day[k]
    print(f"  {k}: {len(df_k)} rows")

## Setup High-Resolution Grid + Simulate Both Fields

In [ ]:
# ---- high-res simulation grid ----
dlat_hr = DELTA_LAT_BASE / HR_LAT_FACTOR
dlon_hr = DELTA_LON_BASE / HR_LON_FACTOR
lats_hr = torch.arange(LAT_RANGE[0] - 0.1, LAT_RANGE[1] + 0.1,
                       dlat_hr, device=DEVICE, dtype=DTYPE)
lons_hr = torch.arange(LON_RANGE[0] - 0.1, LON_RANGE[1] + 0.1,
                       dlon_hr, device=DEVICE, dtype=DTYPE)
print(f"High-res grid: {len(lats_hr)} lat × {len(lons_hr)} lon = {len(lats_hr)*len(lons_hr):,} cells")

# ---- true log-parameter vector (shared by both smooth scenarios) ----
true_log    = true_to_log_params(TRUE_DICT)
true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)
print("True log-params:", [f"{v:.4f}" for v in true_log])

# ---- simulate one GP field per true smoothness ----
# Both use the SAME TRUE_DICT; only the kernel form differs.
fields = {}
for ts in TRUE_SMOOTHS:
    print(f"Simulating field from smooth={ts}...", end=" ")
    t0 = time.time()
    torch.manual_seed(SEED)   # same noise seed for both so only kernel differs
    fields[ts] = generate_field_values(
        lats_hr, lons_hr, T_STEPS, true_params, dlat_hr, dlon_hr, smooth=ts
    )
    print(f"Done in {time.time()-t0:.1f}s | shape={tuple(fields[ts].shape)}")

## Build Resolution Setups (grid, ordering, obs mapping)

In [ ]:
# ---- shared infrastructure: does NOT depend on true_smooth ----
# Each resolution R gives a different target grid and observation match count.

base_setups = {}

for R in RESOLUTION_MULTS:
    dlat = DELTA_LAT_BASE * R
    dlon = DELTA_LON_BASE * R
    print(f"\nx{R}x{R}: spacing={dlat:.4f}° lat × {dlon:.4f}° lon")

    # Regular target grid at this resolution
    lats_g = torch.arange(LAT_RANGE[0], LAT_RANGE[1] + 1e-4, dlat, device=DEVICE, dtype=DTYPE)
    lons_g = torch.arange(LON_RANGE[0], LON_RANGE[1] + 1e-4, dlon, device=DEVICE, dtype=DTYPE)
    lats_g = torch.round(lats_g * 10000) / 10000
    lons_g = torch.round(lons_g * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats_g, lons_g, indexing="ij")
    grid_coords  = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    n_grid       = grid_coords.shape[0]
    print(f"  Grid: {len(lats_g)} lat × {len(lons_g)} lon = {n_grid} cells")

    # Maxmin ordering + NN list (for Vecchia conditioning)
    coords_np         = grid_coords.cpu().numpy()
    ord_grid          = _orderings.maxmin_cpp(coords_np)
    nns_grid          = _orderings.find_nns_l2(locs=coords_np[ord_grid], max_nn=MM_COND_NUMBER)
    ordered_coords_np = coords_np[ord_grid]

    # Precompute obs → grid assignment (independent of field values)
    # Tolerance = half the cell width at this resolution
    s3_list, hr_list, src_list = precompute_mapping_indices(
        ref_day, lats_hr, lons_hr, grid_coords,
        day_keys, dlat_half=dlat / 2, dlon_half=dlon / 2
    )

    n_obs_total = sum(int(np.sum(a >= 0)) for a in s3_list)
    print(f"  Matched obs: {n_obs_total} total across {T_STEPS} steps "
          f"({n_obs_total/T_STEPS:.0f}/step avg)")

    base_setups[R] = {
        "grid_coords":       grid_coords,
        "ord_grid":          ord_grid,
        "nns_grid":          nns_grid,
        "ordered_coords_np": ordered_coords_np,
        "n_grid":            n_grid,
        "s3_list":           s3_list,
        "hr_list":           hr_list,
        "src_list":          src_list,
        "n_obs_total":       n_obs_total,
        "dlat":              dlat,
        "dlon":              dlon,
    }
    gc.collect()

print("\nResolution setups complete.")

## Assemble Irregular Maps (per true_smooth × resolution)

In [ ]:
# irr_maps[(true_smooth, R)] = ordered irregular map for that scenario
# Obs POSITIONS are the same across true_smooth (same grid, same GEMS day);
# obs VALUES differ because the underlying field differs.

irr_maps = {}

for ts in TRUE_SMOOTHS:
    for R in RESOLUTION_MULTS:
        setup   = base_setups[R]
        irr_map = assemble_irregular_map(
            fields[ts],
            setup["s3_list"], setup["hr_list"], setup["src_list"],
            day_keys, setup["grid_coords"], true_params,
        )
        # Apply maxmin ordering so the Vecchia class sees ordered data
        irr_maps[(ts, R)] = {k: v[setup["ord_grid"]] for k, v in irr_map.items()}
        del irr_map
        gc.collect()
        torch.cuda.empty_cache()

print("Irregular maps assembled for all (true_smooth, R) combos:")
for ts in TRUE_SMOOTHS:
    for R in RESOLUTION_MULTS:
        imap = irr_maps[(ts, R)]
        n_obs = sum(
            int((~torch.isnan(v[:, 2])).sum().item())
            for v in imap.values()
        )
        print(f"  true_smooth={ts}, x{R}x{R}: {n_obs} obs")

## Fitting Loop — 16 Fits

In [ ]:
# ---- initial parameter values (same for all 16 fits) ----
rng      = np.random.default_rng(SEED)
init_log = list(true_log)
for idx in [0, 1, 2, 3, 6]:   # log-scale params
    init_log[idx] += rng.uniform(-0.4, 0.4)
for idx in [4, 5]:             # advection (linear scale)
    scale = max(abs(init_log[idx]), 0.05)
    init_log[idx] += rng.uniform(-scale, scale)
init_phys = backmap_params(init_log)
print("Initial params (physical):")
for k in P_LABELS:
    print(f"  {k:12s}: init={init_phys[k]:.4f}  true={TRUE_DICT[k]:.4f}")

In [ ]:
results = []

for ts in TRUE_SMOOTHS:
    for R in RESOLUTION_MULTS:
        setup       = base_setups[R]
        irr_map_ord = irr_maps[(ts, R)]

        for fs in FIT_SMOOTHS:
            correctly_specified = (ts == fs)
            spec_label = "CORRECT" if correctly_specified else "MISSPEC"
            tag = f"true={ts}  fit={fs}  x{R}x{R}  [{spec_label}]"
            print(f"\n{'='*65}")
            print(f"Fitting: {tag}")
            print(f"  n_obs={setup['n_obs_total']}  n_grid={setup['n_grid']}")

            params = [
                torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True)
                for v in init_log
            ]

            model = HybridVecchiaFit(
                smooth=fs,
                input_map=irr_map_ord,
                nns_map=setup["nns_grid"],
                mm_cond_number=MM_COND_NUMBER,
                nheads=NHEADS,
                limit_A=LIMIT_A,
                limit_B_local=LIMIT_B_LOCAL,
                limit_C_local=LIMIT_C_LOCAL,
                daily_stride=DAILY_STRIDE,
                spatial_coords=setup["ordered_coords_np"],
                lag1_lon_offset=LAG1_LON_OFFSET,
                lag1_fresh_count=LAG1_FRESH,
                lag2_fresh_count=LAG2_FRESH,
            )
            model.precompute_conditioning_sets()

            optimizer = model.set_optimizer(
                params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, history_size=LBFGS_HIST
            )
            t0  = time.time()
            out, fit_iter = model.fit_vecc_lbfgs(
                params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5
            )
            elapsed = time.time() - t0

            loss = float(out[-1])
            est  = backmap_params(out)
            print(f"  loss={loss:.4f}  elapsed={elapsed:.1f}s  steps={fit_iter+1}")
            for k in P_LABELS:
                re = abs(est[k] - TRUE_DICT[k]) / max(abs(TRUE_DICT[k]), 0.01)
                print(f"    {k:12s}: est={est[k]:8.4f}  true={TRUE_DICT[k]:8.4f}  re={re:.3f}")

            results.append({
                "true_smooth":          ts,
                "fit_smooth":           fs,
                "correctly_specified":  correctly_specified,
                "R":                    R,
                "n_obs_total":          setup["n_obs_total"],
                "n_grid":               setup["n_grid"],
                "loss":                 loss,
                "fit_steps":            fit_iter + 1,
                "elapsed_s":            round(elapsed, 1),
                **{f"est_{k}": est[k] for k in P_LABELS},
            })
            del model
            gc.collect()
            torch.cuda.empty_cache()

df = pd.DataFrame(results)
print(f"\nDone. {len(df)} fits recorded.")

## Results Summary

In [ ]:
# ---- relative error per parameter ----
for k in P_LABELS:
    tv = TRUE_DICT[k]
    denom = max(abs(tv), 0.01)
    df[f"re_{k}"] = (df[f"est_{k}"] - tv).abs() / denom

re_cols = [f"re_{k}" for k in P_LABELS]
df["overall_rmsre"] = np.sqrt(df[re_cols].pow(2).mean(axis=1))

disp = df[["true_smooth", "fit_smooth", "R", "n_obs_total",
           "overall_rmsre", "loss", "elapsed_s"] + re_cols].copy()
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
print(disp.sort_values(["true_smooth", "fit_smooth", "R"]).to_string(index=False))

In [ ]:
# ---- pivot: overall RMSRE by (true_smooth × fit_smooth) and resolution ----
pivot = df.pivot_table(
    index=["true_smooth", "fit_smooth", "correctly_specified"],
    columns="R",
    values="overall_rmsre",
)
pivot.columns = [f"x{c}x{c}" for c in pivot.columns]
pivot = pivot.reindex(columns=[f"x{r}x{r}" for r in RESOLUTION_MULTS])
print("Overall RMSRE (lower = better):")
print(pivot.to_string())

## Plot 1 — Parameter Trajectories Across Resolutions

In [ ]:
# Visual encoding
# Color: true_smooth (blue=0.5, red=1.5)
# Linestyle: fit_smooth (solid=0.5, dashed=1.5)
# Thickness: correctly specified = thick

COLOR  = {0.5: "steelblue",  1.5: "tomato"}
LS     = {0.5: "-",          1.5: "--"}
LW     = {True: 2.5,          False: 1.5}
MARKER = {0.5: "o",           1.5: "s"}

# Sort by n_obs_total (ascending: x8→x4→x2→x1)
x_vals = sorted(df["n_obs_total"].unique())

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (param, plabel) in enumerate(zip(P_LABELS, P_DISP)):
    ax  = axes[i]
    tv  = TRUE_DICT[param]

    for ts in TRUE_SMOOTHS:
        for fs in FIT_SMOOTHS:
            sub = df[(df.true_smooth == ts) & (df.fit_smooth == fs)]\
                    .sort_values("n_obs_total")
            correct = (ts == fs)
            ax.plot(
                sub["n_obs_total"], sub[f"est_{param}"],
                color=COLOR[ts],
                ls=LS[fs],
                lw=LW[correct],
                marker=MARKER[ts],
                ms=7,
                label=f"true={ts}, fit={fs}",
            )
            # annotate resolution labels on last fit_smooth=0.5 curve
            if fs == 0.5 and ts == 0.5:
                for _, row in sub.iterrows():
                    ax.annotate(
                        f"x{int(row.R)}",
                        (row["n_obs_total"], row[f"est_{param}"]),
                        textcoords="offset points", xytext=(4, 4),
                        fontsize=7, color="grey",
                    )

    ax.axhline(tv, color="black", ls=":", lw=1.5, label=f"true={tv:.3g}")
    ax.set_title(plabel, fontsize=13)
    ax.set_xlabel("Total matched obs (↑ = finer grid)")
    ax.set_ylabel("Estimate")
    ax.legend(fontsize=7, loc="best")
    ax.grid(True, alpha=0.25)

axes[-1].set_visible(False)

# Legend guide
legend_entries = [
    mlines.Line2D([], [], color=COLOR[0.5], ls="-",  lw=2.5, marker="o", label="true=0.5, fit=0.5 [correct]"),
    mlines.Line2D([], [], color=COLOR[0.5], ls="--", lw=1.5, marker="o", label="true=0.5, fit=1.5 [misspec]"),
    mlines.Line2D([], [], color=COLOR[1.5], ls="--", lw=1.5, marker="s", label="true=1.5, fit=0.5 [misspec]"),
    mlines.Line2D([], [], color=COLOR[1.5], ls="-",  lw=2.5, marker="s", label="true=1.5, fit=1.5 [correct]"),
    mlines.Line2D([], [], color="black",   ls=":",  lw=1.5,             label="true value"),
]
axes[-1].legend(handles=legend_entries, loc="center", fontsize=10, frameon=True)
axes[-1].set_visible(True)
axes[-1].axis("off")

fig.suptitle(
    "Parameter Estimates vs Resolution  (x8=coarsest → x1=finest)\n"
    "Thick solid = correctly specified  |  Thin dashed = misspecified",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig("param_trajectories_16fit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: param_trajectories_16fit.png")

## Plot 2 — Model Misspecification Bias (at each resolution)

In [ ]:
# ---- Bias (est - true)/true for each param, grouped by (ts, fs), one panel per resolution ----
fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=False)
bar_w   = 0.18
x_param = np.arange(len(P_LABELS))

combos  = [(0.5, 0.5), (0.5, 1.5), (1.5, 0.5), (1.5, 1.5)]
bar_colors = [COLOR[0.5], COLOR[0.5], COLOR[1.5], COLOR[1.5]]
bar_alpha  = [0.9, 0.45, 0.45, 0.9]
bar_hatch  = ["", "//", "//", ""]
bar_label  = ["ts=0.5,fs=0.5", "ts=0.5,fs=1.5", "ts=1.5,fs=0.5", "ts=1.5,fs=1.5"]

for ax_i, R in enumerate(RESOLUTION_MULTS):
    ax = axes[ax_i]
    n_obs = base_setups[R]["n_obs_total"]

    for ci, (ts, fs) in enumerate(combos):
        row = df[(df.R == R) & (df.true_smooth == ts) & (df.fit_smooth == fs)]
        if row.empty:
            continue
        row = row.iloc[0]
        # signed relative bias: (est - true) / |true|
        bias = np.array([
            (row[f"est_{k}"] - TRUE_DICT[k]) / max(abs(TRUE_DICT[k]), 0.01)
            for k in P_LABELS
        ])
        offset = (ci - 1.5) * bar_w
        ax.bar(
            x_param + offset, bias,
            width=bar_w,
            color=bar_colors[ci],
            alpha=bar_alpha[ci],
            hatch=bar_hatch[ci],
            edgecolor="black",
            linewidth=0.5,
            label=bar_label[ci],
        )

    ax.axhline(0, color="black", lw=1.2)
    ax.set_xticks(x_param)
    ax.set_xticklabels(P_DISP, fontsize=9, rotation=25)
    ax.set_title(f"Resolution x{R}x{R}\n({n_obs} obs)", fontsize=11)
    ax.set_ylabel("(est − true) / |true|" if ax_i == 0 else "")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2, axis="y")

fig.suptitle(
    "Signed Relative Bias by Parameter and Resolution\n"
    "Solid fill = true_smooth matches fit_smooth (correct)  |  Hatched = misspecified",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.savefig("misspec_bias_by_resolution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: misspec_bias_by_resolution.png")

## Plot 3 — Relative Error Heatmap (quick robustness overview)

In [ ]:
import matplotlib.colors as mcolors

# Rows: (true_smooth, fit_smooth, R) combinations
# Cols: parameters
combo_labels = []
re_matrix    = []

for ts in TRUE_SMOOTHS:
    for fs in FIT_SMOOTHS:
        for R in RESOLUTION_MULTS:
            row = df[(df.R == R) & (df.true_smooth == ts) & (df.fit_smooth == fs)]
            if row.empty:
                continue
            row = row.iloc[0]
            re_vec = [row[f"re_{k}"] for k in P_LABELS]
            spec   = "✓" if ts == fs else "✗"
            combo_labels.append(f"ts={ts} fs={fs} x{R} {spec}")
            re_matrix.append(re_vec)

re_matrix = np.array(re_matrix)

fig, ax = plt.subplots(figsize=(12, max(4, len(combo_labels) * 0.45)))
im = ax.imshow(re_matrix, aspect="auto", cmap="RdYlGn_r",
               vmin=0, vmax=min(re_matrix.max(), 1.5))
plt.colorbar(im, ax=ax, label="Relative Error |est−true|/|true|")
ax.set_xticks(range(len(P_LABELS)))
ax.set_xticklabels(P_DISP, fontsize=11)
ax.set_yticks(range(len(combo_labels)))
ax.set_yticklabels(combo_labels, fontsize=9)

# Annotate cells
for r in range(re_matrix.shape[0]):
    for c in range(re_matrix.shape[1]):
        val = re_matrix[r, c]
        ax.text(c, r, f"{val:.2f}", ha="center", va="center",
                fontsize=7.5,
                color="white" if val > 0.8 else "black")

ax.set_title("Relative Error Heatmap  (✓=correctly specified, ✗=misspecified)",
             fontsize=12)
plt.tight_layout()
plt.savefig("re_heatmap_16fit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: re_heatmap_16fit.png")

## Summary Analysis

In [ ]:
print("=" * 70)
print("RESOLUTION SENSITIVITY: Does more data improve estimation?")
print("=" * 70)
for ts in TRUE_SMOOTHS:
    for fs in FIT_SMOOTHS:
        sub = df[(df.true_smooth == ts) & (df.fit_smooth == fs)].sort_values("n_obs_total")
        spec = "CORRECT" if ts == fs else "MISSPEC"
        print(f"\ntrue_smooth={ts}, fit_smooth={fs} [{spec}]")
        print(f"  {'n_obs':>8}  {'RMSRE':>8}  {'range_lon':>10}  {'nugget':>8}  {'advec_lon':>10}")
        for _, row in sub.iterrows():
            print(f"  {int(row.n_obs_total):>8}  "
                  f"{row.overall_rmsre:>8.4f}  "
                  f"{row.est_range_lon:>10.4f}  "
                  f"{row.est_nugget:>8.4f}  "
                  f"{row.est_advec_lon:>10.4f}")

print("\n" + "=" * 70)
print("MISSPECIFICATION EFFECT: Range bias (at finest resolution x1)")
print("Expected: wrong smooth → range bias (compensates for different spatial smoothness)")
print("=" * 70)
finest = df[df.R == 1]
for _, row in finest.iterrows():
    spec = "CORRECT" if row.correctly_specified else "MISSPEC"
    print(f"ts={row.true_smooth}, fs={row.fit_smooth} [{spec}]: "
          f"range_lon={row.est_range_lon:.4f} (true={TRUE_DICT['range_lon']:.4f})  "
          f"nugget={row.est_nugget:.4f} (true={TRUE_DICT['nugget']:.4f})  "
          f"RMSRE={row.overall_rmsre:.4f}")

print("\n" + "=" * 70)
print("ROBUSTNESS: Which parameters are most sensitive to resolution?")
print("(Range across all 16 fits per parameter)")
print("=" * 70)
for k in P_LABELS:
    col   = f"re_{k}"
    rng   = df[col].max() - df[col].min()
    mean_ = df[col].mean()
    print(f"  {k:12s}: mean_RE={mean_:.4f}  range_RE={rng:.4f} "
          + ("  [SENSITIVE]" if rng > 0.3 else ""))